# 灰色关联分析 (Grey Relational Analysis, GRA)

## 算法简介

灰色关联分析是灰色系统理论中的重要方法，由邓聚龙教授于1982年提出。
其基本思想是：**通过比较各方案与理想方案（参考序列）的几何相似程度来评价方案的优劣**。

关联度越大，说明该方案与理想方案越接近，方案越优。

## 算法步骤

1. **指标正向化**：将所有类型指标统一转化为极大型指标
2. **无量纲化**：消除各指标量纲影响（均值化或初值化）
3. **确定参考序列**：选取各指标的最优值组成理想方案
4. **计算绝对差矩阵**：$\Delta_{ij} = |x_{0j} - x_{ij}|$
5. **计算灰色关联系数**：$\xi_{ij} = \frac{\min\Delta + \rho \cdot \max\Delta}{\Delta_{ij} + \rho \cdot \max\Delta}$
6. **计算关联度并排序**：$r_i = \sum w_j \cdot \xi_{ij}$

## 步骤 0：准备数据

假设有 3 个评价方案，2 个评价指标。
- 指标 1：产量（极大型，越高越好）
- 指标 2：成本（极小型，越低越好）

In [ ]:
import numpy as np

# 原始评价矩阵：3 个方案 × 2 个指标
# 方案1: 产量=8, 成本=5
# 方案2: 产量=7, 成本=3
# 方案3: 产量=9, 成本=4
matrix = np.array([
    [8.0, 5.0],
    [7.0, 3.0],
    [9.0, 4.0],
])

n, m = matrix.shape
print(f"方案数 n = {n}, 指标数 m = {m}")
print(f"原始矩阵:\n{matrix}")

## 步骤 1：指标正向化

将所有指标统一转化为极大型指标（越大越好）。
- 极大型（类型 1）：无需转换
- 极小型（类型 2）：$x' = \max(x) - x$ 或 $x' = 1/x$
- 中间型（类型 3）：$x' = 1 - \frac{|x - x_{best}|}{\max|x - x_{best}|}$
- 区间型（类型 4）：落在区间内为 1，否则按距离衰减

In [ ]:
kinds = [1, 2]  # 指标1=极大型, 指标2=极小型

# 对极小型指标（第2列）进行正向化：max - x
converted = matrix.copy()
max_col2 = np.max(matrix[:, 1])
converted[:, 1] = max_col2 - matrix[:, 1]

print(f"正向化后的矩阵（所有指标均为极大型）:\n{converted}")
print(f"\n解释：成本列（原值 {matrix[:, 1].tolist()}）→ 正向化后 {converted[:, 1].tolist()}")

## 步骤 2：无量纲化处理

消除各指标的量纲影响，使数据具有可比性。这里使用**均值化**方法：每列除以该列均值。

In [ ]:
# 均值化：每列除以该列均值
col_mean = np.mean(converted, axis=0)
normalized = converted / col_mean

print(f"各列均值：{np.round(col_mean, 4).tolist()}")
print(f"无量纲化后矩阵（每列均值为 1）:\n{np.round(normalized, 4)}")
print(f"\n验证：各列均值 = {np.round(np.mean(normalized, axis=0), 4).tolist()}")

## 步骤 3：确定参考序列（母序列）

参考序列即理想方案：每个指标取最优值（正向化后就是最大值）。

In [ ]:
# 参考序列 = 每列最大值
reference = np.max(normalized, axis=0)

print(f"参考序列（理想方案）：{np.round(reference, 4).tolist()}")
print(f"\n含义：每个指标在所有方案中的最优值")

## 步骤 4：计算绝对差矩阵

$\Delta_{ij} = |x_{0j} - x_{ij}|$，即每个方案各指标与参考序列对应值的绝对差距。

In [ ]:
diff = np.abs(normalized - reference)

print(f"绝对差矩阵 Δ:\n{np.round(diff, 4)}")
print(f"\n两极最小差 min(Δ) = {np.min(diff):.4f}")
print(f"两极最大差 max(Δ) = {np.max(diff):.4f}")

## 步骤 5：计算灰色关联系数矩阵

$$\xi_{ij} = \frac{\min_i\min_j \Delta_{ij} + \rho \cdot \max_i\max_j \Delta_{ij}}{\Delta_{ij} + \rho \cdot \max_i\max_j \Delta_{ij}}$$

其中 $\rho \in (0, 1]$ 为**分辨系数**，通常取 0.5。$\rho$ 越小，分辨能力越强。

In [ ]:
rho = 0.5  # 分辨系数

min_diff = np.min(diff)
max_diff = np.max(diff)

coefficients = (min_diff + rho * max_diff) / (diff + rho * max_diff)

print(f"分辨系数 ρ = {rho}")
print(f"\n关联系数矩阵 ξ:\n{np.round(coefficients, 4)}")
print(f"\n关联系数范围：[{np.min(coefficients):.4f}, {np.max(coefficients):.4f}]")

## 步骤 6：计算灰色关联度并排序

关联度为各方案关联系数的加权平均：
$$r_i = \sum_{j=1}^{m} w_j \cdot \xi_{ij}$$

关联度越大，方案越优。

In [ ]:
# 等权重
weights = np.ones(m) / m

# 计算关联度
grades = np.sum(coefficients * weights, axis=1)

# 排序（关联度越大越优）
ranks = np.argsort(-grades) + 1

print(f"权重：{np.round(weights, 4).tolist()}")
print(f"\n{'方案':<6}{'关联度 r':<14}{'排名':<6}")
print("-" * 26)
for i in range(n):
    rank_i = int(np.where(ranks == i + 1)[0][0]) + 1
    print(f"{i + 1:<6}{grades[i]:<14.4f}{rank_i:<6}")

best = int(np.argmax(grades) + 1)
print(f"\n最优方案：方案 {best}")

## 验证：使用封装好的模块

使用 `src.algorithms.grey_relational_analysis` 模块进行一站式计算，验证结果一致性。

In [ ]:
import sys
from pathlib import Path

_project_root = Path.cwd()
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from src.algorithms.grey_relational_analysis import grey_relational_analysis

result = grey_relational_analysis(matrix, kinds=kinds, rho=0.5)

print(f"模块计算结果：")
print(f"  关联度：{np.round(result['grades'], 4).tolist()}")
print(f"  排名：  {result['ranks'].tolist()}")
print(f"\n与手动计算一致：{np.allclose(grades, result['grades'])}")

## 分辨系数 ρ 的影响分析

ρ 越小，关联系数之间的差异越大，分辨能力越强。下面比较不同 ρ 值的效果。

In [ ]:
rho_values = [0.1, 0.3, 0.5, 0.7, 1.0]

print(f"{'ρ':<8}{'关联度':<40}{'最优方案':<10}")
print("-" * 58)
for rho_val in rho_values:
    result = grey_relational_analysis(matrix, kinds=kinds, rho=rho_val)
    grades_rounded = np.round(result['grades'], 4).tolist()
    best_idx = int(np.argmax(result['grades']) + 1)
    print(f"{rho_val:<8}{str(grades_rounded):<40}{best_idx:<10}")

## 使用初值化方法

当初值不为零时，也可以使用初值化（每列除以第一个值），适用于时间序列等有序数据。

In [ ]:
result_init = grey_relational_analysis(matrix, kinds=kinds, normalize_method="init")

print(f"初值化后的矩阵（第一行全为 1）：\n{np.round(result_init['normalized'], 4)}")
print(f"\n关联度：{np.round(result_init['grades'], 4).tolist()}")
print(f"排名：  {result_init['ranks'].tolist()}")